# Main task

In [49]:
from s3_utils import connect_s3, download_s3, upload_s3
import requests
from prefect import flow, task
from prefect.blocks.system import Secret
import datetime
import base64
import io
import zipfile
import json
import pandas as pd


@task(name = "Fetch Legiscan Datasets List")
def fetch_legiscan_dataset_list(legiscan_api_key):
    """
    Fetches list of currently available session datasets for all legislative bodies in current year.
    """
    current_year = str(datetime.date.today().year)
    dataset_list_url = f'https://api.legiscan.com/?key={legiscan_api_key}&op=getDatasetList&year={current_year}&state=CA'
    response = requests.get(dataset_list_url)
    return response.json()


def format_append_bill(bill_file, bill, bills_dataframe):
    """
    Formats a bill record and appends to bills dataframe.
    """

    status_dict = {
        1: 'Introduced',
        2: 'Engrossed',
        3: 'Enrolled',
        4: 'Passed',
        5: 'Vetoed',
        6: 'Failed'
    }

    bill = {
        'bill_file' : bill_file,
        'legislative_body' : bill['state'],
        'bill_number' : bill['bill_number'],
        'url' : bill['url'],
        'status' : status_dict.get(bill['status']),
        'status_date' : bill['status_date'],
        'title' : bill['title'],
        'description' : bill['description'],   
        'sponsors' : [f"{sponsor['name']} ({sponsor['party']})" for sponsor in bill['sponsors']]
    }

    bills_dataframe = pd.concat([bills_dataframe, pd.DataFrame([bill])])

    return bills_dataframe


@task(name = "Upload tagged bills to S3")
def upload_tagged_bills(bills_dataframe):
    s3 = connect_s3('portfolio-project-files')
    upload_s3(s3, 'sdg-bill-tracking/tagged_bills.csv', bills_dataframe)
    


@task(name = "Fetch Legiscan Datesets")
def fetch_legiscan_datasets(legiscan_api_key, datasets):
    """
    Iterates available session datasets returned from fetch_legiscan_dataset_list()
    Pulls Zip file for each session.
    Parses bill-related datasets from each Zip file.
    Returns as one master JSON.
    """
    
    bills_dataframe = pd.DataFrame()

    for dataset in datasets['datasetlist']:

        # Fetch Zip file for state / session
        session_id = dataset['session_id']
        access_key = dataset['access_key']
        url = f'https://api.legiscan.com/?key={legiscan_api_key}&op=getDataset&id={session_id}&access_key={access_key}'
        response = requests.get(url)

        # Parse Zip file for bill datasets
        b64_zip = response.json()['dataset']['zip']
        zip_bytes = base64.b64decode(b64_zip)
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
            bill_files = [f for f in z.namelist() if '/bill/' in f]

            # Iterate bill files
            for bill_file in bill_files:
                with z.open(bill_file) as f:

                    # Format & append to bill dataframe
                    bill = json.load(f)['bill']
                    bills_dataframe = format_append_bill(bill_file, bill, bills_dataframe)

    return bills_dataframe.dropna().reset_index(drop = True)


@flow(name = 'Bill Tagging Main', log_prints = True)
async def bill_processing_main():

    # Authentication
    legiscan_api_key = await Secret.load("legiscan-api-key")
    legiscan_api_key = legiscan_api_key.get()

    # Fetch session datasets
    datasets = fetch_legiscan_dataset_list(legiscan_api_key)

    # Fetch bill datasets and append to master dataframe
    bills_dataframe = fetch_legiscan_datasets(legiscan_api_key, datasets)

    # Apply tags to bills
    bills_dataframe = bill_tagging_main(bills_dataframe)

    # Upload to S3
    upload_tagged_bills(bills_dataframe)

    return bills_dataframe

bills_dataframe = await bill_processing_main()

14:30:02.651 | INFO    | prefect.engine - View at https://app.prefect.cloud/account/3f9bd1dc-a34b-4ba7-a6e0-e2aa163f25d6/workspace/4fe3e874-7162-4c73-879d-d1f78dbe5925/runs/flow-run/06993703-aada-7dd7-8000-228c688908ea

14:30:03.033 | INFO    | Flow run 'nifty-hyrax' - Beginning flow run 'nifty-hyrax' for flow 'Bill Tagging Main'

14:30:03.038 | INFO    | Flow run 'nifty-hyrax' - View at https://app.prefect.cloud/account/3f9bd1dc-a34b-4ba7-a6e0-e2aa163f25d6/workspace/4fe3e874-7162-4c73-879d-d1f78dbe5925/runs/flow-run/06993703-aada-7dd7-8000-228c688908ea

14:30:03.550 | INFO    | Task run 'Fetch Legiscan Datasets List-3f5' - Finished in state Completed()

14:30:09.029 | INFO    | Task run 'Fetch Legiscan Datesets-8e3' - Finished in state Completed()

14:30:10.517 | INFO    | Task run 'Get SDG Corpus-5fc' - Finished in state Completed()

14:30:24.413 | INFO    | Task run 'Tag new bills-4c3' - Finished in state Completed()

14:30:24.419 | INFO    | Task run 'Bill tagging main-fd3' - Finished in state Completed()

14:30:26.078 | INFO    | Task run 'Upload tagged bills to S3-147' - Finished in state Completed()

14:30:26.247 | INFO    | Flow run 'nifty-hyrax' - Finished in state Completed()

In [47]:
bills_dataframe[bills_dataframe.tagged_sdgs.apply(lambda x: len(x) == 0)]

,bill_file,legislative_body,bill_number,url,status,status_date,title,description,sponsors,tagged_sdgs,tagged_targets
30,CA/2025-2026_Regular_Session/bill/AB16.json,CA,AB16,https://legiscan.com/CA/bill/AB16/2025,Passed,2025-10-01,Vote by mail ballots: processing.,An act to amend Sections 15101 and 15104 of th...,[Juan Alanis (R)],[],[]
31,CA/2025-2026_Regular_Session/bill/AB17.json,CA,AB17,https://legiscan.com/CA/bill/AB17/2025,Passed,2025-07-30,Elections: precinct maps.,An act to add Section 12263 to the Elections C...,[Juan Alanis (R)],[],[]
38,CA/2025-2026_Regular_Session/bill/AB24.json,CA,AB24,https://legiscan.com/CA/bill/AB24/2025,Failed,2026-02-02,San Diego Association of Governments: board of...,An act to amend Section 132351.1 of the Public...,[Carl DeMaio (R)],[],[]
41,CA/2025-2026_Regular_Session/bill/AB27.json,CA,AB27,https://legiscan.com/CA/bill/AB27/2025,Engrossed,2025-05-29,Personal Income Tax Law: Corporation Tax Law: ...,An act to add Sections 17157.5 and 24309.9 to ...,"[Pilar Schiavo (D), Benjamin Allen (D), Steve ...",[],[]
43,CA/2025-2026_Regular_Session/bill/AB29.json,CA,AB29,https://legiscan.com/CA/bill/AB29/2025,Failed,2026-02-02,Medi-Cal: Adverse Childhood Experiences trauma...,An act to add Section 14105.198 to the Welfare...,[Joaquin Arambula (D)],[],[]
...,...,...,...,...,...,...,...,...,...,...,...
3590,CA/2025-2026_Regular_Session/bill/ACA7.json,CA,ACA7,https://legiscan.com/CA/bill/ACA7/2025,Introduced,2025-02-13,Government preferences.,A resolution to propose to the people of the S...,[Corey Jackson (D)],[],[]
3591,CA/2025-2026_Regular_Session/bill/ACA8.json,CA,ACA8,https://legiscan.com/CA/bill/ACA8/2025,Passed,2025-08-21,Congressional redistricting.,A resolution to propose to the people of the S...,"[Robert Rivas (D), Mike McGuire (D), Dawn Addi...",[],[]
3593,CA/2025-2026_Regular_Session/bill/ACA10.json,CA,ACA10,https://legiscan.com/CA/bill/ACA10/2025,Introduced,2025-03-05,Parole.,A resolution to propose to the people of the S...,[Carl DeMaio (R)],[],[]
3602,CA/2025-2026_Regular_Session/bill/SCA2.json,CA,SCA2,https://legiscan.com/CA/bill/SCA2/2025,Introduced,2025-02-10,Governor: pardons and commutations.,A resolution to propose to the people of the S...,"[Steven Choi (R), Marie Alvarado-Gil (R), Leti...",[],[]


# Tagging task

In [97]:
from ordered_set import OrderedSet
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np


@task(name = "Get SDG Corpus")
def get_sdg_corpus():
    """
    Downloads SDG corpus from S3 and formats.
    """
    s3 = connect_s3('portfolio-project-files')
    SDGS = download_s3(s3, 'sdg-bill-tracking/sdg_indicators_corpus.csv')
    SDGS = SDGS.groupby(['SDG No.', 'Target No.', 'SDG', 'Target'])['Indicator'].apply(lambda x: '\n'.join(x)).reset_index().reset_index(drop = True).rename(columns = {'Indicator' : 'Indicators'})
    return SDGS


@task(name = "Get Tagged Bills")
def get_tagged_bills():
    """
    Downloads Tagged Bills from S3.
    Purpose: To avoid re-processing tags for older bills.
    """
    s3 = connect_s3('portfolio-project-files')
    tagged_bills = download_s3(s3, 'sdg-bill-tracking/tagged_bills.csv')
    tagged_bills = tagged_bills[['bill_file', 'tagged_sdgs', 'tagged_targets']].set_index('bill_file').to_dict(orient = 'index')
    return tagged_bills


@task(name = 'Tag new bills')
def tag_new_bills(bills_dataframe, SDGS, threshold=0.3, top_k=3):

    # Generate embeddings
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v1')
    SDG_embeddings = model.encode(SDGS.apply(lambda row: '\n'.join(list(OrderedSet([row['Target'].strip(), row['Indicators'].strip()]))), axis = 1))
    SDGS['embedding'] = list(SDG_embeddings)
    bill_embeddings = model.encode(bills_dataframe.apply(lambda row: '\n'.join(list(OrderedSet([row['title'].strip(), row['description'].strip()]))), axis = 1))
    bills_dataframe['embedding'] = list(bill_embeddings)

    # Stack embeddings
    sdg_matrix = np.vstack(SDGS["embedding"].values)
    bill_matrix = np.vstack(bills_dataframe["embedding"].values)

    # Compute cosine similarity
    similarity_matrix = cosine_similarity(bill_matrix, sdg_matrix)

    tagged_sdgs = []
    tagged_targets = []

    for row in similarity_matrix:
        # Get indices sorted descending
        sorted_idx = np.argsort(row)[::-1]

        # Keep only those above threshold
        filtered = [
            idx for idx in sorted_idx
            if row[idx] >= threshold
        ][:top_k]

        if filtered:
            tagged_sdgs.append(list(set(SDGS.iloc[filtered]["SDG No."])))
            tagged_targets.append(list(set(SDGS.iloc[filtered]["Target No."])))
        else:
            tagged_sdgs.append([])
            tagged_targets.append([])

    bills_dataframe["tagged_sdgs"] = tagged_sdgs
    bills_dataframe["tagged_targets"] = tagged_targets
    return bills_dataframe.drop(columns = 'embedding')


@task(name = "Bill tagging main")
def bill_tagging_main(bills_dataframe):

    # Fetch current tagged bills and SDG corpus from S3
    SDGS = get_sdg_corpus()
    tagged_bills = get_tagged_bills()

    # Apply tags where already available in S3
    for col in ['tagged_sdgs', 'tagged_targets']:
        bills_dataframe[col] = bills_dataframe.bill_file.apply(lambda x: tagged_bills.get(x, {}).get(col))

    # Apply tags to new bills
    bills_dataframe_new = bills_dataframe[pd.isna(bills_dataframe.tagged_sdgs)].reset_index(drop = True)
    if len(bills_dataframe_new) > 0:
        bills_dataframe_new = tag_new_bills(bills_dataframe_new, SDGS)

    # Append results and return
    bills_dataframe = pd.concat([bills_dataframe_new, bills_dataframe])
    return bills_dataframe.drop_duplicates(subset = 'bill_file', keep = 'first').reset_index(drop = True)


In [98]:
bills_dataframe = bill_tagging_main(bills_dataframe)

14:48:17.469 | INFO    | Task run 'Get SDG Corpus' - Finished in state Completed()

14:48:23.246 | INFO    | Task run 'Get Tagged Bills' - Finished in state Completed()

14:48:23.260 | INFO    | Task run 'Bill tagging main' - Finished in state Completed()

In [99]:
bills_dataframe

,bill_file,legislative_body,bill_number,url,status,status_date,title,description,sponsors,tagged_sdgs,tagged_targets
0,CA/2025-2026_Regular_Session/bill/AB36.json,CA,AB36,https://legiscan.com/CA/bill/AB36/2025,Passed,2025-10-10,Housing elements: prohousing designation.,An act to amend Section 65589.9 of the Governm...,[Esmeralda Soria (D)],"[11, 5]","['11.1', '5.4']"
1,CA/2025-2026_Regular_Session/bill/AB37.json,CA,AB37,https://legiscan.com/CA/bill/AB37/2025,Failed,2026-02-02,Furnishing hypodermic needles and syringes.,An act to amend Section 4145.5 of the Business...,[Sade Elhawary (D)],[3],['3.b']
2,CA/2025-2026_Regular_Session/bill/AB38.json,CA,AB38,https://legiscan.com/CA/bill/AB38/2025,Failed,2026-02-02,Crimes: serious and violent felonies.,An act to amend Section 667.5 of the Penal Cod...,[Tom Lackey (R)],[16],"['16.4', '16.3']"
3,CA/2025-2026_Regular_Session/bill/AB39.json,CA,AB39,https://legiscan.com/CA/bill/AB39/2025,Passed,2025-10-06,General plans: Local Electrification Planning ...,An act to add Section 65302.13 to the Governme...,"[Rick Zbur (D), Benjamin Allen (D), Henry Ster...","[16, 13, 6]","['13.2', '16.6', '6.b']"
4,CA/2025-2026_Regular_Session/bill/AB40.json,CA,AB40,https://legiscan.com/CA/bill/AB40/2025,Engrossed,2025-04-21,Redistricting: congressional districts.,An act to amend Section 21454 of the Elections...,"[Marc Berman (D), Isaac Bryan (D)]",[],[]
...,...,...,...,...,...,...,...,...,...,...,...
3600,CA/2025-2026_Regular_Session/bill/AB31.json,CA,AB31,https://legiscan.com/CA/bill/AB31/2025,Engrossed,2025-06-02,Peace officers: tribal police pilot project.,An act to add and repeal Sections 830.83 and 8...,[James Ramos (D)],"[8, 10, 5]","['5.1', '8.8', '10.3']"
3601,CA/2025-2026_Regular_Session/bill/AB32.json,CA,AB32,https://legiscan.com/CA/bill/AB32/2025,Failed,2026-02-02,Tribal judges.,An act to amend Section 2166.7 of the Election...,"[Esmeralda Soria (D), Blanca Pacheco (D), Jame...",[16],"['16.7', '16.5']"
3602,CA/2025-2026_Regular_Session/bill/AB33.json,CA,AB33,https://legiscan.com/CA/bill/AB33/2025,Engrossed,2025-05-29,Autonomous vehicles.,"An act to add Sections 38760, 38761, and 38762...","[Cecilia Aguiar-Curry (D), Tom Lackey (R), Daw...",[],[]
3603,CA/2025-2026_Regular_Session/bill/AB34.json,CA,AB34,https://legiscan.com/CA/bill/AB34/2025,Engrossed,2026-01-26,California Renewables Portfolio Standard Progr...,An act to amend Section 399.30 of the Public U...,"[Joe Patterson (R), Josh Becker (D), Marc Berm...",[7],"['7.1', '7.2', '7.b']"
